## Agentic AI pattern: LLM as a judge ##

In [7]:
import os
import sys
import json
import requests
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display

%load_ext autoreload
%autoreload 2
import agentix

print(f'Package version: {agentix.__version__}')
print(f'Authors:         {agentix.__authors__}')
print(f'Python version:  {sys.version}')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Package version: 0.0.1
Authors:         Andreas Werdich
The Core for Computational Biomedicine at Harvard Medical School
https://dbmi.hms.harvard.edu/about-dbmi/core-computational-biomedicine
Python version:  3.12.13 (main, Jul 23 2026, 14:43:28) [Clang 22.1.3 ]


### Azure OpenAI ###

In [8]:
load_dotenv()
api_key = os.environ['AZURE_API_KEY']
base_url = f'https://azure-ai.hms.edu/openai/v1'
openai = OpenAI(base_url=base_url, api_key=api_key, default_headers={'api-key': api_key})

In [9]:
request = """
Please come up with a challenging, nuanced question with a succinct answer,
that I can ask a number of LLMs to evaluate their intelligence.
Not a mathematical puzzle, but more of a thought-provoking question that requires intelligent insight.
Include in your question that the answer must be short.
"""
request += "Answer only with the question, no explanation."
messages = [{"role": "user", "content": request}]

In [10]:
response = openai.chat.completions.create(model='gpt-5', messages=messages)
question = response.choices[0].message.content
display(Markdown(question))

Which widely debated false dichotomy misguides policy or public discourse, and how would you reframe it into a more useful question? Answer in one sentence (max 20 words).

In [11]:
competitors = []
answers = []
messages = [{"role": "user", "content": question}]

def record(model_name, answer):
    competitors.append(model_name)
    answers.append(answer)
    display(Markdown(answer))

In [12]:
# Let's record some responses from commercial models

model_name = 'gpt-5'

response = openai.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

record(model_name, answer)

Economic growth vs environmental protection is a false dichotomy; ask: how can we align prosperity with rapid decarbonization and restoration?

In [13]:
display(dict(zip(competitors, answers)))

{'gpt-5': 'Economic growth vs environmental protection is a false dichotomy; ask: how can we align prosperity with rapid decarbonization and restoration?'}

### Ollama ###

In [14]:
ollama_url = 'http://localhost:11434'
OLLAMA_BASE_URL = f'{ollama_url}/v1'
print(requests.get(ollama_url).content)
models = requests.get(f'{ollama_url}/v1/models').json()
for model in models.get('data'):
    print(model.get('id'))
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

b'Ollama is running'
gemma4:31b
gemma4:e4b
qwen2.5:7b
llama3.2:latest


In [15]:
# Responses from the Ollama model

model_name = 'gemma4:e4b'

response = ollama.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

record(model_name, answer)

How can we redesign economies to ensure sustainable prosperity while prioritizing environmental health?

In [16]:
display(dict(zip(competitors, answers)))

{'gpt-5': 'Economic growth vs environmental protection is a false dichotomy; ask: how can we align prosperity with rapid decarbonization and restoration?',
 'gemma4:e4b': 'How can we redesign economies to ensure sustainable prosperity while prioritizing environmental health?'}

In [17]:
together = ''
for index, answer in enumerate(answers):
    together += f'# Response from competitor {index + 1}\n\n'
    together += answer + '\n\n'

In [18]:
print(together)

# Response from competitor 1

Economic growth vs environmental protection is a false dichotomy; ask: how can we align prosperity with rapid decarbonization and restoration?

# Response from competitor 2

How can we redesign economies to ensure sustainable prosperity while prioritizing environmental health?




### LLM as a judge ##

In [19]:
judge = f'''

You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:

{{'results': ['best competitor number', 'second best competitor number', 'third best competitor number', ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting.'''

In [20]:
display(Markdown(judge))



You are judging a competition between 2 competitors.
Each model has been given this question:

Which widely debated false dichotomy misguides policy or public discourse, and how would you reframe it into a more useful question? Answer in one sentence (max 20 words).

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:

{'results': ['best competitor number', 'second best competitor number', 'third best competitor number', ...]}

Here are the responses from each competitor:

# Response from competitor 1

Economic growth vs environmental protection is a false dichotomy; ask: how can we align prosperity with rapid decarbonization and restoration?

# Response from competitor 2

How can we redesign economies to ensure sustainable prosperity while prioritizing environmental health?



Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting.

In [21]:
judge_messages = [{'role': 'user', 'content': judge}]

In [22]:
# Judgement time!
model_name = 'gpt-5'
response = openai.chat.completions.create(model=model_name, messages=judge_messages)

In [23]:
results = response.choices[0].message.content
print(results)

{"results": ["1", "2"]}


In [24]:
results_dict = json.loads(results)
ranks = results_dict['results']
for index, result in enumerate(ranks):
    competitor = competitors[int(result)-1]
    print(f'Rank {index + 1}: {competitor}')

Rank 1: gpt-5
Rank 2: gemma4:e4b
